# Chapter 10~11. 정책기반 강화학습과 PPO — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter08_reinforce_ppo.ipynb)

책 본문: [Chapter 10](https://smhanlab.com/book-ml/kor/ml2/chapter10.html) / [Chapter 11](https://smhanlab.com/book-ml/kor/ml2/chapter11.html)

REINFORCE가 실제로 좋은 행동의 확률을 밀어올리는 과정과, PPO의
클리핑이 어떤 모양의 목적함수를 만드는지 직접 실행해서 확인합니다.

## 1. REINFORCE: 정말로 좋은 행동을 더 선호하게 되는가?

장난감 2-행동 밴딧: action 0을 고르면 보상 +1, action 1을 고르면
보상 -1 (상태는 항상 동일, `state_feature=1.0`). 정책이 이 사실을
스스로 배우는지 봅니다.

In [ ]:
import math, random

def softmax_policy(theta, state_feature):
    logits = [theta[0] * state_feature, theta[1] * state_feature]
    m = max(logits)
    exps = [math.exp(l - m) for l in logits]
    total = sum(exps)
    return [e / total for e in exps]

def reinforce_update(theta, episode, alpha, gamma):
    T = len(episode)
    G = [0.0] * T
    running = 0.0
    for t in reversed(range(T)):
        running = episode[t][2] + gamma * running
        G[t] = running
    for t, (s, a, r) in enumerate(episode):
        probs = softmax_policy(theta, s)
        if a == 0:
            grad_log_pi = [(1 - probs[0]) * s, -probs[1] * s]
        else:
            grad_log_pi = [-probs[0] * s, (1 - probs[1]) * s]
        theta[0] += alpha * G[t] * grad_log_pi[0]
        theta[1] += alpha * G[t] * grad_log_pi[1]
    return theta

In [ ]:
random.seed(0)
theta = [0.0, 0.0]  # 처음엔 두 행동을 반반으로 선택 (softmax(0,0) = [0.5, 0.5])
state_feature = 1.0

for episode_idx in range(300):
    a = 0 if random.random() < softmax_policy(theta, state_feature)[0] else 1
    r = 1.0 if a == 0 else -1.0
    episode = [(state_feature, a, r)]
    theta = reinforce_update(theta, episode, alpha=0.05, gamma=0.99)

final_probs = softmax_policy(theta, state_feature)
print(f"학습된 theta = {theta}")
print(f"action 0(좋은 행동)을 고를 확률 = {final_probs[0]:.4f}")
assert final_probs[0] > 0.9, "충분히 학습됐다면 action 0을 90% 이상 선호해야 합니다"
print("보상이 더 큰 행동(action 0)의 확률이 실제로 크게 올라갔습니다.")

## 2. PPO 클리핑: 목적함수가 실제로 어떤 모양인가

\\(L^{CLIP}(\theta) = \mathbb{E}_t[\min(r_t A_t,\ \text{clip}(r_t, 1-\epsilon,
1+\epsilon) A_t)]\\) 를 여러 \\(r_t\\) 값에 대해 계산해서, 어드밴티지가
양수/음수일 때 각각 그래프가 어디서 평평해지는지(그래디언트가
사라지는지) 직접 확인합니다.

In [ ]:
def ppo_clip_loss(ratio, advantage, epsilon=0.2):
    unclipped = ratio * advantage
    clipped = max(min(ratio, 1 + epsilon), 1 - epsilon) * advantage
    return min(unclipped, clipped)

ratios = [0.5 + 0.01 * i for i in range(151)]  # 0.5 ~ 2.0
loss_pos_adv = [ppo_clip_loss(r, advantage=1.0) for r in ratios]
loss_neg_adv = [ppo_clip_loss(r, advantage=-1.0) for r in ratios]

print(f"advantage=+1, r=0.5  -> L^CLIP = {ppo_clip_loss(0.5, 1.0):.3f}")
print(f"advantage=+1, r=1.5  -> L^CLIP = {ppo_clip_loss(1.5, 1.0):.3f}  (1+eps=1.2에서 이미 꺾임)")
print(f"advantage=-1, r=0.5  -> L^CLIP = {ppo_clip_loss(0.5, -1.0):.3f}  (1-eps=0.8에서 이미 꺾임)")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(ratios, loss_pos_adv, label="advantage = +1 (좋은 행동)")
ax.plot(ratios, loss_neg_adv, label="advantage = -1 (나쁜 행동)")
ax.axvline(0.8, color="gray", linestyle="--", linewidth=1)
ax.axvline(1.2, color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("probability ratio r_t(theta)")
ax.set_ylabel("L^CLIP")
ax.set_title("PPO clipped objective: r=0.8/1.2를 넘으면 평평해짐(그래디언트 소실)")
ax.legend()
plt.show()